# 06 — Importancia de variables
**Doctorado en Informática – UTN FRRe/UNNE/UNAM**  
Curso: Introducción al Aprendizaje de Máquina  
Integrantes: Stella Gerzel · Juan Manuel Goya · Gustavo Ebel  
**Responsable: Gustavo Ebel**

---
En este notebook quise ver qué variables son las que más peso tienen en las predicciones del modelo. Para eso usé dos métodos y comparé los resultados.

El dataset de entrada es `DataSet_modelado_base.xlsx`.

In [ ]:
# librerías que voy a necesitar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print('Librerías OK')

In [ ]:
# cargo el dataset que preparó Stella en el notebook 03
# ya viene limpio — solo tengo que separar el target y hacer el split

from google.colab import files
uploaded = files.upload()

ruta = next(iter(uploaded))
df = pd.read_excel(ruta, sheet_name='Datos_modelado_base')
df_w = df.copy()

# Separar variables predictoras del target
X_base = df_w.drop(columns=['nota']).copy()

# Definir 3 targets
y1 = df_w['nota'].map({'A':0,'R':1,'U':1})
mask_p2 = df_w['nota'].isin(['A','R'])
X_p2 = X_base[mask_p2].reset_index(drop=True)
y2   = df_w.loc[mask_p2,'nota'].map({'A':0,'R':1}).reset_index(drop=True)
mask_p3 = df_w['nota'].isin(['A','U'])
X_p3 = X_base[mask_p3].reset_index(drop=True)
y3   = df_w.loc[mask_p3,'nota'].map({'A':0,'U':1}).reset_index(drop=True)

# Splits estratificados 75/25
from sklearn.model_selection import train_test_split
X1_tr,X1_te,y1_tr,y1_te = train_test_split(X_base,y1,test_size=0.25,stratify=y1,random_state=42)
X2_tr,X2_te,y2_tr,y2_te = train_test_split(X_p2,  y2,test_size=0.25,stratify=y2,random_state=42)
X3_tr,X3_te,y3_tr,y3_te = train_test_split(X_p3,  y3,test_size=0.25,stratify=y3,random_state=42)

# Preprocesador — mismo criterio que notebook 03 del informe:
# numéricas → mediana + StandardScaler
# categóricas → 'Sin dato' + OneHotEncoder
from sklearn.preprocessing import StandardScaler
cols_cat = X_base.select_dtypes('object').columns.tolist()
cols_num = X_base.select_dtypes('number').columns.tolist()

print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Variables predictoras: {X_base.shape[1]} ({len(cols_cat)} categóricas, {len(cols_num)} numéricas)')
print(f'Train P1:{len(X1_tr)} | P2:{len(X2_tr)} | P3:{len(X3_tr)}')


In [ ]:
# CELDA 3 — Función auxiliar: preprocesar + entrenar RF para un problema
#
# Clave: cada problema tiene su PROPIO preprocesador ajustado solo
# a su propio X_tr. Los datos transformados se guardan como arrays numpy
# separados del pipeline, eliminando el problema de incompatibilidad.

params = {
    'P1 Riesgo global': dict(n_estimators=200, max_depth=None, min_samples_leaf=1),
    'P2 Desaprobacion': dict(n_estimators=100, max_depth=None, min_samples_leaf=1),
    'P3 Abandono':      dict(n_estimators=200, max_depth=20,   min_samples_leaf=1),
}

problemas = [
    ('P1 Riesgo global', X1_tr, X1_te, y1_tr, y1_te),
    ('P2 Desaprobacion', X2_tr, X2_te, y2_tr, y2_te),
    ('P3 Abandono',      X3_tr, X3_te, y3_tr, y3_te),
]

# Guardar: modelo RF, datos transformados, nombres columnas originales
datos_por_problema = {}

for nombre, X_tr, X_te, y_tr, y_te in problemas:
    # Preprocesador propio para este problema
    prep = ColumnTransformer([
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('enc', OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False))]), cols_cat),
        ('num', SimpleImputer(strategy='most_frequent'), cols_num)
    ])

    # Transformar: fit SOLO en train, transform en train y test
    X_tr_t = prep.fit_transform(X_tr)
    X_te_t = prep.transform(X_te)        # usa los parámetros del train

    # Verificar consistencia ANTES de entrenar
    assert X_tr_t.shape[1] == X_te_t.shape[1], \
        f'ERROR: train {X_tr_t.shape[1]} cols vs test {X_te_t.shape[1]} cols'

    # Entrenar RF directamente sobre datos ya transformados
    rf = RandomForestClassifier(class_weight='balanced', random_state=42,
                                **params[nombre])
    rf.fit(X_tr_t, y_tr)

    auc = roc_auc_score(y_te, rf.predict_proba(X_te_t)[:,1])

    # Nombres de columnas después del one-hot (para mapear de vuelta)
    enc = prep.named_transformers_['cat']['enc']
    nombres_cols = enc.get_feature_names_out(cols_cat).tolist() + cols_num

    datos_por_problema[nombre] = {
        'rf': rf, 'prep': prep,
        'X_tr_t': X_tr_t, 'X_te_t': X_te_t,
        'X_te': X_te, 'y_te': y_te,
        'nombres_cols': nombres_cols, 'auc': auc
    }
    print(f'{nombre}: {X_tr_t.shape[1]} columnas codificadas | ROC-AUC test = {auc:.3f}')

print('\nTodos los modelos entrenados correctamente.')

In [ ]:
# calculo la importancia interna y la agrupo por variable original
# (el one-hot la divide en muchas columnas, tengo que sumar)

importancias_internas = {}

for nombre, datos in datos_por_problema.items():
    rf           = datos['rf']
    nombres_cols = datos['nombres_cols']
    imp          = rf.feature_importances_

    # Agrupar por variable original sumando aportes de sus columnas one-hot
    imp_dict = {}
    for var_orig in X_base.columns.tolist():
        total = sum(v for n, v in zip(nombres_cols, imp)
                    if n.startswith(var_orig + '_') or n == var_orig)
        if total > 0:
            imp_dict[var_orig] = total

    imp_serie = pd.Series(imp_dict).sort_values(ascending=False)
    importancias_internas[nombre] = imp_serie

    print(f'{nombre} — Top 10:')
    print(imp_serie.head(10).round(4).to_string())
    print()

# Gráfico
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (nombre, imp_serie) in zip(axes, importancias_internas.items()):
    top15 = imp_serie.head(15)
    ax.barh(top15.index[::-1], top15.values[::-1], color='#3498DB', alpha=0.85)
    ax.set_title(f'{nombre}\n(importancia interna RF)', fontweight='bold')
    ax.set_xlabel('Importancia acumulada')
plt.suptitle('Top 15 — Importancia interna de Random Forest',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# permutation importance manual
# desordeno los valores de cada variable y mido cuánto cae el AUC
# trabajo con los arrays numpy directamente para evitar problemas con el pipeline

print('Calculando permutation importance (puede tardar 1-2 minutos)...\n')

N_REPEATS = 5
rng = np.random.RandomState(42)
perm_importancias = {}

for nombre, datos in datos_por_problema.items():
    rf           = datos['rf']
    X_te_t       = datos['X_te_t'].copy()
    y_te         = datos['y_te']
    nombres_cols = datos['nombres_cols']
    auc_base     = datos['auc']

    # Índices de columnas transformadas para cada variable original
    indices_var = {}
    for var_orig in X_base.columns.tolist():
        idx = [i for i, n in enumerate(nombres_cols)
               if n.startswith(var_orig + '_') or n == var_orig]
        if idx:
            indices_var[var_orig] = idx

    resultados = []
    for var_orig, idx in indices_var.items():
        caidas = []
        for _ in range(N_REPEATS):
            X_perm = X_te_t.copy()
            # Desordenar todas las columnas de esta variable juntas
            perm_idx = rng.permutation(X_perm.shape[0])
            X_perm[:, idx] = X_perm[perm_idx][:, idx]
            auc_perm = roc_auc_score(y_te, rf.predict_proba(X_perm)[:,1])
            caidas.append(auc_base - auc_perm)
        resultados.append({
            'variable':    var_orig,
            'importancia': np.mean(caidas),
            'std':         np.std(caidas)
        })

    perm_df = (pd.DataFrame(resultados)
                 .sort_values('importancia', ascending=False)
                 .reset_index(drop=True))
    perm_importancias[nombre] = perm_df

    print(f'{nombre} — Top 10 (caída en ROC-AUC):')
    print(perm_df[perm_df['importancia']>0].head(10)
          [['variable','importancia','std']].round(4).to_string(index=False))
    print()

In [ ]:
# CELDA 6 — Gráfico permutation importance
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (nombre, perm_df) in zip(axes, perm_importancias.items()):
    top15 = perm_df[perm_df['importancia']>0].head(15)
    ax.barh(top15['variable'][::-1], top15['importancia'][::-1],
            xerr=top15['std'][::-1], color='#E74C3C', alpha=0.85, capsize=4)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'{nombre}\n(permutation importance)', fontweight='bold')
    ax.set_xlabel('Caída en ROC-AUC al desordenar')
plt.suptitle('Top 15 — Permutation importance',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
---
## Resumen — variables más relevantes

Acá comparo los dos métodos y me quedo con las variables que aparecen en el top 10 de los dos. Esas son las más confiables para mencionar en el informe.